In [1]:
!pip install ipywidgets -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')
print("✅ All libraries loaded!")

display(HTML("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@700;900&family=Rajdhani:wght@400;600&display=swap');
  .banner {
    background: linear-gradient(135deg, #0f0c29, #1a1040, #0d1f3c);
    border-radius: 16px; padding: 40px 30px; text-align: center;
    border: 1px solid rgba(251,146,60,0.3);
    box-shadow: 0 0 40px rgba(251,146,60,0.15), inset 0 0 60px rgba(0,0,0,0.4);
    margin-bottom: 10px;
  }
  .banner h1 {
    font-family: 'Orbitron', monospace; font-size: 1.8em; font-weight: 900;
    background: linear-gradient(90deg, #fb923c, #fbbf24, #34d399);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    margin: 0 0 8px 0; letter-spacing: 2px;
  }
  .banner h2 {
    font-family: 'Orbitron', monospace; font-size: 1em; font-weight: 700;
    color: #fb923c; margin: 0 0 6px 0; letter-spacing: 1px;
  }
  .banner p { font-family: 'Rajdhani', sans-serif; color: #94a3b8; font-size: 1em; margin: 0; }
  .step-row { display: flex; justify-content: center; gap: 12px; margin-top: 20px; flex-wrap: wrap; }
  .step-card {
    background: rgba(251,146,60,0.08); border: 1px solid rgba(251,146,60,0.2);
    border-radius: 10px; padding: 10px 18px;
    font-family: 'Rajdhani', sans-serif; color: #cbd5e1; font-size: 0.9em;
  }
  .step-card span { color: #fb923c; font-weight: 700; margin-right: 6px; }
  .algo-row { display: flex; justify-content: center; gap: 10px; margin-top: 16px; flex-wrap: wrap; }
  .algo-badge {
    background: rgba(251,146,60,0.1); border: 1px solid rgba(251,146,60,0.3);
    border-radius: 8px; padding: 6px 14px;
    font-family: 'Rajdhani', sans-serif; color: #fb923c; font-size: 0.82em;
    letter-spacing: 0.5px;
  }
  .section-title {
    font-family: 'Orbitron', monospace;
    background: linear-gradient(135deg, #0f0c29, #1a1040);
    color: #fb923c; padding: 14px 22px; border-radius: 10px;
    font-size: 0.9em; border-left: 4px solid #fb923c;
    margin: 14px 0; letter-spacing: 1px;
  }
</style>

<div class="banner">
  <h2>📈 MODEL 3</h2>
  <h1>Temporal Student Progression Learning Model</h1>
  <p>Temporal Student Performance Progression Model — Gradient Boosted Decision Trees</p>
  <div class="algo-row">
    <div class="algo-badge">📊 Gradient Boosted Decision Trees</div>
    <div class="algo-badge">⏱️ Feature Alignment Sem1→Sem2</div>
    <div class="algo-badge">📈 Performance Delta Learning</div>
    <div class="algo-badge">🔁 Progress / Decline Tracking</div>
  </div>
  <div class="step-row">
    <div class="step-card"><span>①</span>Upload</div>
    <div class="step-card"><span>②</span>Train</div>
    <div class="step-card"><span>③</span>Predict</div>
    <div class="step-card"><span>④</span>Dashboard</div>
    <div class="step-card"><span>⑤</span>Metrics</div>
    <div class="step-card"><span>⑥</span>Download</div>
  </div>
</div>
"""))

display(HTML('<div class="section-title">📂 STEP 1 — Upload Your Data Files</div>'))

upload_btn1 = widgets.Button(description='📁 Upload sem1.csv',
    style={'button_color': '#c2410c'},
    layout=widgets.Layout(width='220px', height='42px'))
upload_btn2 = widgets.Button(description='📁 Upload sem 2.csv',
    style={'button_color': '#b45309'},
    layout=widgets.Layout(width='220px', height='42px'))

status1 = widgets.Label(value='⏳ Not uploaded')
status2 = widgets.Label(value='⏳ Not uploaded')

sem1, sem2, sem2_original = None, None, None

def upload_sem1(b):
    global sem1
    print("📂 Choose sem1.csv ↓")
    files.upload()
    sem1 = pd.read_csv("sem1.csv")
    status1.value = f'✅ sem1.csv — {sem1.shape[0]} students, {sem1.shape[1]} cols'

def upload_sem2(b):
    global sem2, sem2_original
    print("📂 Choose sem 2.csv ↓")
    files.upload()
    sem2 = pd.read_csv("sem 2.csv")
    sem2_original = sem2.copy()
    status2.value = f'✅ sem 2.csv — {sem2.shape[0]} students, {sem2.shape[1]} cols'

upload_btn1.on_click(upload_sem1)
upload_btn2.on_click(upload_sem2)
display(widgets.HBox([upload_btn1, status1]))
display(widgets.HBox([upload_btn2, status2]))

sem1_pairs = [
    ("Operating System mid term cia(30)",              "Operating System Final (70)"),
    ("Data Structures and Algorithms MidSem cia (30)", "Data Structures and Algorithms Final (70)"),
    ("maths  MidSem (30)",                             "maths Final (70)"),
    ("DBMS MidSem cia (30)",                           "DBMS Final (70)"),
    ("DS PRACTICAL(15)",                               "DS PRACTICAL FINAL(35)"),
    ("DBMS PRACTICAL (15)",                            "DBMS PRACTICAL FINAL (35)"),
    ("JAVA PRACTICAL (15)",                            "JAVA PRACTICAL FINAL (35)"),
    ("Problem Solving Techniques PRACTICAL (15)",      "Problem Solving Techniques FINAL (35)"),
    ("Java MidSem (15)",                               "Java final (35)")
]

sem2_subjects = [
    "Degn & Analysis of Algorithms (30)",
    "Artificial Intelligence (30)",
    "Computer Networks - Theory (30)",
    "Software Engineering Methodologies - Theory (30)"
]
lab_subjects = [
    "Computer Networks - Lab (15)",
    "Software Engineering Methodologies - Lab (15)",
    "Web Technologies - Lab (15)",
    "Python Lab (15)"
]

def clean_marks(df):
    df = df.copy()
    for col in df.columns:
        df[col] = df[col].astype(str).str.replace("+","",regex=False).str.strip()
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def grade(c):
    if c >= 9:   return "O"
    elif c >= 8: return "A+"
    elif c >= 7: return "A"
    elif c >= 6: return "B+"
    elif c >= 5: return "B"
    else:        return "F"

def style_ax(ax, title):
    ax.set_facecolor('#1e293b')
    ax.set_title(title, color='white', fontsize=10, fontweight='bold', pad=8)
    ax.tick_params(colors='#94a3b8', labelsize=8)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.xaxis.label.set_color('#94a3b8')
    ax.yaxis.label.set_color('#94a3b8')

print("✅ Subjects and helpers defined!")

display(HTML('<div class="section-title">⚙️ STEP 2 — Train Temporal Progression Model</div>'))

train_btn = widgets.Button(description='🚀 Train Model 3',
    style={'button_color': '#c2410c'},
    layout=widgets.Layout(width='220px', height='44px'))
train_out = widgets.Output()

M3_model  = None
model_mae = None
model_rmse= None
model_r2  = None
model_acc = None
delta_stats = {}

def run_training(b):
    global M3_model, model_mae, model_rmse, model_r2, model_acc, delta_stats
    global sem1, sem2, sem2_original

    with train_out:
        clear_output()
        if sem1 is None or sem2 is None:
            display(HTML('<p style="color:#f87171;font-family:monospace">❌ Upload both files first!</p>'))
            return

        sem1_c = clean_marks(sem1).fillna(0)
        sem2_c = clean_marks(sem2).fillna(0)
        sem2_original = sem2_c.copy()

        display(HTML('<p style="color:#fb923c;font-family:monospace;font-weight:bold">📈 Building temporal delta features...</p>'))

        # ── STEP 1: Build delta features ──
        # Delta = Final - Internal per subject (captures improvement/decline)
        delta_list    = []
        internal_list = []
        final_list    = []

        for inp, out in sem1_pairs:
            internal = sem1_c[inp].values
            final    = sem1_c[out].values
            delta    = final - internal   # performance progression

            internal_list.append(pd.Series(internal))
            delta_list.append(pd.Series(delta))
            final_list.append(pd.Series(final))

        internal_all = pd.concat(internal_list, ignore_index=True)
        delta_all    = pd.concat(delta_list,    ignore_index=True)
        final_all    = pd.concat(final_list,    ignore_index=True)

        # Store delta statistics for visualization
        delta_stats = {
            "mean":  float(delta_all.mean()),
            "std":   float(delta_all.std()),
            "min":   float(delta_all.min()),
            "max":   float(delta_all.max()),
            "positive_pct": float((delta_all > 0).mean() * 100)
        }

        display(HTML(f'''
        <div style="background:#1e293b;border-radius:10px;padding:14px 18px;
                    border:1px solid rgba(251,146,60,0.3);margin:8px 0;
                    font-family:monospace;font-size:0.9em">
          <span style="color:#94a3b8">📊 Delta stats: </span>
          <span style="color:#fb923c">Mean={delta_stats["mean"]:.2f}</span> |
          <span style="color:#fbbf24">Std={delta_stats["std"]:.2f}</span> |
          <span style="color:#34d399">Min={delta_stats["min"]:.2f}</span> |
          <span style="color:#f87171">Max={delta_stats["max"]:.2f}</span> |
          <span style="color:#60a5fa">Improved={delta_stats["positive_pct"]:.1f}%</span>
        </div>'''))

        # ── STEP 2: Feature matrix ──
        X = pd.DataFrame({
            "Internal": internal_all,
            "Delta":    delta_all       # key temporal feature
        })
        y = final_all

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        display(HTML('<p style="color:#fb923c;font-family:monospace;font-weight:bold">🌳 Training Gradient Boosted Decision Trees...</p>'))

        # ── STEP 3: Train GBDT ──
        M3_model = GradientBoostingRegressor(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            random_state=42
        )
        M3_model.fit(X_train, y_train)

        preds = M3_model.predict(X_test)
        model_mae  = mean_absolute_error(y_test, preds)
        model_rmse = np.sqrt(mean_squared_error(y_test, preds))
        model_r2   = r2_score(y_test, preds)
        model_acc  = (1 - model_mae / 70) * 100

        display(HTML(f'''
        <div style="background:linear-gradient(135deg,#1e293b,#0f172a);border-radius:12px;
                    padding:18px 20px;border:1px solid rgba(52,211,153,0.3);margin-top:10px">
          <p style="color:#34d399;font-family:monospace;font-size:1em;margin:0 0 8px 0;font-weight:bold">
            ✅ Model 3 Trained Successfully!
          </p>
          <div style="display:flex;gap:20px;flex-wrap:wrap;font-family:monospace;font-size:0.9em">
            <span style="color:#94a3b8">MAE: <span style="color:#fb923c;font-weight:bold">{model_mae:.2f}</span></span>
            <span style="color:#94a3b8">RMSE: <span style="color:#fbbf24;font-weight:bold">{model_rmse:.2f}</span></span>
            <span style="color:#94a3b8">R²: <span style="color:#60a5fa;font-weight:bold">{model_r2:.4f}</span></span>
            <span style="color:#94a3b8">Accuracy: <span style="color:#34d399;font-weight:bold">{model_acc:.2f}%</span></span>
          </div>
        </div>'''))

        # ── Training chart ──
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.patch.set_facecolor('#0f172a')

        # Feature importance
        ax = axes[0]; style_ax(ax, "Feature Importance")
        importances = M3_model.feature_importances_
        feat_names  = ["Internal marks", "Delta (progress)"]
        colors_fi   = ["#60a5fa", "#fb923c"]
        bars = ax.bar(feat_names, importances, color=colors_fi, width=0.5, edgecolor='none')
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f'{bar.get_height():.3f}', ha='center', color='white', fontsize=9, fontweight='bold')
        ax.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)

        # Delta distribution
        ax = axes[1]; style_ax(ax, "Performance Delta Distribution")
        ax.hist(delta_all, bins=20, color='#fb923c', edgecolor='#0f172a', alpha=0.9, linewidth=1)
        ax.axvline(0, color='#f87171', linestyle='--', linewidth=1.5, label='Zero delta')
        ax.axvline(delta_stats["mean"], color='#34d399', linestyle='--',
                   linewidth=1.5, label=f'Mean: {delta_stats["mean"]:.1f}')
        ax.set_xlabel("Delta (Final − Internal)")
        ax.set_ylabel("Count")
        ax.legend(fontsize=7, facecolor='#1e293b', labelcolor='white')
        ax.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)

        # Predicted vs Actual scatter
        ax = axes[2]; style_ax(ax, "Predicted vs Actual")
        ax.scatter(y_test, preds, color='#fb923c', alpha=0.6, s=20, edgecolors='none')
        mn = min(float(y_test.min()), float(preds.min()))
        mx = max(float(y_test.max()), float(preds.max()))
        ax.plot([mn, mx], [mn, mx], color='#34d399', linewidth=1.5, linestyle='--', label='Perfect fit')
        ax.set_xlabel("Actual marks")
        ax.set_ylabel("Predicted marks")
        ax.legend(fontsize=7, facecolor='#1e293b', labelcolor='white')

        plt.tight_layout(pad=2); plt.show()

train_btn.on_click(run_training)
display(train_btn, train_out)

display(HTML('<div class="section-title">🔮 STEP 3 — Predict Semester 2 Results</div>'))

predict_btn = widgets.Button(description='⚡ Run Predictions',
    style={'button_color': '#7c3aed'},
    layout=widgets.Layout(width='220px', height='44px'))
predict_out = widgets.Output()
pred_df = None

def run_predictions(b):
    global pred_df, sem2, sem2_original

    with predict_out:
        clear_output()
        if M3_model is None:
            display(HTML('<p style="color:#f87171;font-family:monospace">❌ Train model first!</p>'))
            return

        sem2_c = clean_marks(sem2).fillna(0)
        df     = sem2_original.copy()
        total_cols = []

        display(HTML('<p style="color:#fb923c;font-family:monospace">📈 Predicting using temporal progression model...</p>'))

        # ── Theory subjects ──
        for col in sem2_subjects:
            internal_vals = sem2_c[col].values if col in sem2_c.columns else np.zeros(len(sem2_c))
            # Sem2 has no historical delta yet — use zero (neutral progression)
            delta_neutral = np.zeros(len(sem2_c))
            X_pred = pd.DataFrame({"Internal": internal_vals, "Delta": delta_neutral})
            preds  = np.clip(M3_model.predict(X_pred), 0, 70)
            df[col+"_Final_70"] = preds
            total = col+"_Total_100"
            df[total] = internal_vals + preds
            total_cols.append(total)

        # ── Lab subjects ──
        for col in lab_subjects:
            internal_vals = sem2_c[col].values if col in sem2_c.columns else np.zeros(len(sem2_c))
            delta_neutral = np.zeros(len(sem2_c))
            X_pred = pd.DataFrame({"Internal": internal_vals, "Delta": delta_neutral})
            preds  = np.clip(M3_model.predict(X_pred) * (35/70), 0, 35)
            df[col+"_Final_35"] = preds
            total = col+"_Total_50"
            df[total] = internal_vals + preds
            total_cols.append(total)

        # ── Totals ──
        df["Overall_Total_650"] = df[total_cols].sum(axis=1)
        df["CGPA"]   = (df["Overall_Total_650"] / 650 * 10).round(2)
        df["Result"] = "Pass"

        fail_count = max(1, sem1["Result"].value_counts().get("Fail", 1)) if "Result" in sem1.columns else 1
        df.loc[df.nsmallest(fail_count, "Overall_Total_650").index, "Result"] = "Fail"
        df["Grade"] = df["CGPA"].apply(grade)

        pred_df = df.copy()
        pred_df.to_csv("M3_Temporal_Predictions.csv", index=False)

        display(HTML(f'''
        <div style="background:linear-gradient(135deg,#1e293b,#0f172a);border-radius:12px;
                    padding:16px 20px;border:1px solid rgba(52,211,153,0.3);margin-top:10px">
          <p style="color:#34d399;font-family:monospace;font-size:1.1em;margin:0">
            ✅ Temporal Model predictions complete!<br>
            📊 {len(df)} students |
            ✅ Pass: {(df["Result"]=="Pass").sum()} |
            ❌ Fail: {(df["Result"]=="Fail").sum()} |
            📈 Avg CGPA: {df["CGPA"].mean():.2f} |
            🏆 Top CGPA: {df["CGPA"].max():.2f}
          </p>
        </div>'''))

predict_btn.on_click(run_predictions)
display(predict_btn, predict_out)

display(HTML('<div class="section-title">📊 STEP 4 — Analytics Dashboard</div>'))

dash_btn = widgets.Button(description='📈 Show Dashboard',
    style={'button_color': '#0e7490'},
    layout=widgets.Layout(width='220px', height='44px'))
dash_out = widgets.Output()

def show_dashboard(b):
    with dash_out:
        clear_output()
        if pred_df is None:
            display(HTML('<p style="color:#f87171;font-family:monospace">❌ Run predictions first!</p>'))
            return

        df = pred_df.copy()
        name_col = next((c for c in df.columns if "name" in c.lower()), df.columns[0])

        total_s   = len(df)
        pass_s    = (df["Result"]=="Pass").sum()
        fail_s    = (df["Result"]=="Fail").sum()
        avg_cgpa  = df["CGPA"].mean()
        top_cgpa  = df["CGPA"].max()
        top_name  = df.loc[df["CGPA"].idxmax(), name_col]
        avg_total = df["Overall_Total_650"].mean()

        # ── Summary Cards ──
        display(HTML(f"""
        <style>
          .cards{{display:flex;gap:12px;flex-wrap:wrap;margin:14px 0}}
          .card{{flex:1;min-width:110px;background:linear-gradient(135deg,#1e293b,#0f172a);
                 border-radius:12px;padding:16px 10px;
                 border:1px solid rgba(251,146,60,0.2);text-align:center}}
          .card .num{{font-family:'Orbitron',monospace;font-size:1.5em;font-weight:900;line-height:1.1}}
          .card .lbl{{font-family:'Rajdhani',sans-serif;color:#64748b;font-size:0.75em;margin-top:4px;letter-spacing:1px}}
        </style>
        <div class="cards">
          <div class="card"><div class="num" style="color:#60a5fa">{total_s}</div><div class="lbl">TOTAL STUDENTS</div></div>
          <div class="card"><div class="num" style="color:#34d399">{pass_s}</div><div class="lbl">PASSED</div></div>
          <div class="card"><div class="num" style="color:#f87171">{fail_s}</div><div class="lbl">FAILED</div></div>
          <div class="card"><div class="num" style="color:#a78bfa">{avg_cgpa:.2f}</div><div class="lbl">AVG CGPA</div></div>
          <div class="card"><div class="num" style="color:#fbbf24">{top_cgpa:.2f}</div>
            <div class="lbl">TOP CGPA<br><span style="font-size:0.6em;color:#94a3b8">{top_name}</span></div></div>
          <div class="card"><div class="num" style="color:#fb923c">{avg_total:.0f}</div><div class="lbl">AVG TOTAL/650</div></div>
        </div>"""))

        # ── BIG 3×3 Dashboard ──
        fig = plt.figure(figsize=(18, 14))
        fig.patch.set_facecolor('#0f172a')
        fig.suptitle("📈 Temporal Student Progression Model — Semester 2 Dashboard",
                     fontsize=15, fontweight="bold", color='white', y=1.01)

        # 1. Grade Distribution
        ax1 = fig.add_subplot(3, 3, 1)
        style_ax(ax1, "Grade Distribution")
        grade_order  = ["O","A+","A","B+","B","F"]
        grade_colors = ["#34d399","#60a5fa","#a78bfa","#fb923c","#fbbf24","#f87171"]
        grade_counts = df["Grade"].value_counts().reindex(grade_order, fill_value=0)
        bars1 = ax1.bar(grade_counts.index, grade_counts.values,
                        color=grade_colors, edgecolor='#0f172a', width=0.6)
        for bar in bars1:
            if bar.get_height() > 0:
                ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                         str(int(bar.get_height())), ha='center', color='white', fontsize=9, fontweight='bold')
        ax1.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)

        # 2. Pass vs Fail Pie
        ax2 = fig.add_subplot(3, 3, 2)
        ax2.set_facecolor('#1e293b')
        ax2.set_title("Pass vs Fail", color='white', fontsize=10, fontweight='bold', pad=8)
        rc = df["Result"].value_counts()
        wc = ["#34d399" if l == "Pass" else "#f87171" for l in rc.index]
        wedges, texts, autotexts = ax2.pie(
            rc.values, labels=rc.index, colors=wc, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'#0f172a','linewidth':2},
            textprops={'color':'white','fontsize':9}
        )
        for at in autotexts: at.set_fontweight('bold')

        # 3. CGPA Distribution
        ax3 = fig.add_subplot(3, 3, 3)
        style_ax(ax3, "CGPA Distribution")
        ax3.hist(df["CGPA"], bins=12, color='#fb923c', edgecolor='#0f172a', alpha=0.9)
        ax3.axvline(df["CGPA"].mean(), color='#f87171', linestyle='--',
                    linewidth=1.8, label=f'Mean: {df["CGPA"].mean():.2f}')
        ax3.set_xlabel("CGPA"); ax3.set_ylabel("Students")
        ax3.legend(fontsize=8, facecolor='#1e293b', labelcolor='white')
        ax3.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)

        # 4. Top 10 Students
        ax4 = fig.add_subplot(3, 3, 4)
        style_ax(ax4, "Top 10 Students by CGPA")
        top10 = df.nlargest(10, "CGPA")[[name_col,"CGPA"]].reset_index(drop=True)
        top10[name_col] = top10[name_col].astype(str).str.split().str[0]
        cmap = plt.cm.RdYlGn
        norm = mcolors.Normalize(top10["CGPA"].min(), top10["CGPA"].max())
        bar_c4 = [cmap(norm(v)) for v in top10["CGPA"]]
        ax4.barh(top10[name_col], top10["CGPA"], color=bar_c4, edgecolor='#0f172a')
        ax4.set_xlabel("CGPA"); ax4.invert_yaxis()
        ax4.xaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)
        for i, (_, row) in enumerate(top10.iterrows()):
            ax4.text(row["CGPA"]+0.02, i, f"{row['CGPA']:.2f}",
                     va='center', color='white', fontsize=8)

        # 5. Theory Subject Avg Predicted Finals
        ax5 = fig.add_subplot(3, 3, 5)
        style_ax(ax5, "Avg Predicted Final — Theory (out of 70)")
        subj_cols  = [c+"_Final_70" for c in sem2_subjects if c+"_Final_70" in df.columns]
        subj_means = [df[c].mean() for c in subj_cols]
        subj_lbls  = ["DAA","AI","CN","SEM"][:len(subj_cols)]
        bars5 = ax5.bar(subj_lbls, subj_means,
                        color=["#5C6BC0","#29B6F6","#26C6DA","#66BB6A"][:len(subj_cols)],
                        edgecolor='#0f172a', width=0.55)
        ax5.set_ylim(0, 75); ax5.set_ylabel("Average Marks")
        ax5.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)
        for i, v in enumerate(subj_means):
            ax5.text(i, v+0.5, f"{v:.1f}", ha='center', color='white', fontsize=9, fontweight='bold')

        # 6. Lab Subject Avg Predicted Finals
        ax6 = fig.add_subplot(3, 3, 6)
        style_ax(ax6, "Avg Predicted Final — Lab (out of 35)")
        lab_cols  = [c+"_Final_35" for c in lab_subjects if c+"_Final_35" in df.columns]
        lab_means = [df[c].mean() for c in lab_cols]
        lab_lbls  = ["CN Lab","SEM Lab","Web Tech","Python"][:len(lab_cols)]
        ax6.bar(lab_lbls, lab_means,
                color=["#FFA726","#FF7043","#AB47BC","#42A5F5"][:len(lab_cols)],
                edgecolor='#0f172a', width=0.55)
        ax6.set_ylim(0, 40); ax6.set_ylabel("Average Marks")
        ax6.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)
        for i, v in enumerate(lab_means):
            ax6.text(i, v+0.2, f"{v:.1f}", ha='center', color='white', fontsize=9, fontweight='bold')

        # 7. Overall Total Distribution
        ax7 = fig.add_subplot(3, 3, 7)
        style_ax(ax7, "Overall Total Distribution (out of 650)")
        ax7.hist(df["Overall_Total_650"], bins=15, color='#7E57C2',
                 edgecolor='#0f172a', alpha=0.9)
        ax7.axvline(df["Overall_Total_650"].mean(), color='#f87171', linestyle='--',
                    linewidth=1.8, label=f'Mean: {df["Overall_Total_650"].mean():.0f}')
        ax7.set_xlabel("Total Marks"); ax7.set_ylabel("Students")
        ax7.legend(fontsize=8, facecolor='#1e293b', labelcolor='white')
        ax7.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)

        # 8. Delta Distribution (unique to temporal model)
        ax8 = fig.add_subplot(3, 3, 8)
        style_ax(ax8, "Training Delta Distribution (Progress Trend)")
        if delta_stats:
            sem1_c = clean_marks(sem1).fillna(0)
            all_deltas = []
            for inp, out in sem1_pairs:
                all_deltas.extend((sem1_c[out] - sem1_c[inp]).tolist())
            all_deltas = np.array(all_deltas)
            ax8.hist(all_deltas, bins=20, color='#fb923c', edgecolor='#0f172a', alpha=0.85)
            ax8.axvline(0, color='#f87171', linestyle='--', linewidth=1.5, label='No change')
            ax8.axvline(np.mean(all_deltas), color='#34d399', linestyle='--',
                        linewidth=1.5, label=f'Mean Δ={np.mean(all_deltas):.1f}')
            ax8.set_xlabel("Delta (Final − Internal)")
            ax8.set_ylabel("Count")
            ax8.legend(fontsize=7, facecolor='#1e293b', labelcolor='white')
            ax8.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)

        # 9. Feature Importance
        ax9 = fig.add_subplot(3, 3, 9)
        style_ax(ax9, "GBDT Feature Importance")
        if M3_model:
            importances = M3_model.feature_importances_
            feat_names  = ["Internal marks", "Delta (progression)"]
            bars9 = ax9.bar(feat_names, importances,
                            color=["#60a5fa","#fb923c"], width=0.5, edgecolor='#0f172a')
            for bar in bars9:
                ax9.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                         f'{bar.get_height():.3f}', ha='center',
                         color='white', fontsize=9, fontweight='bold')
            ax9.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)

        plt.tight_layout(pad=2.5)
        plt.savefig("M3_dashboard.png", bbox_inches="tight", dpi=150, facecolor='#0f172a')
        plt.show()
        print("📸 Dashboard saved: M3_dashboard.png")

        # ── Top 10 Leaderboard ──
        top10_full = df[[name_col,"CGPA","Grade","Result","Overall_Total_650"]]\
                       .sort_values("CGPA", ascending=False).head(10)
        medal = ["🥇","🥈","🥉"] + ["🎖️"]*7
        rows  = ""
        for i, (_, row) in enumerate(top10_full.iterrows()):
            gc  = {"O":"#34d399","A+":"#60a5fa","A":"#a78bfa",
                   "B+":"#fb923c","B":"#fbbf24","F":"#f87171"}.get(row["Grade"],"white")
            rc2 = "#34d399" if row["Result"]=="Pass" else "#f87171"
            rows += f"""<tr style="border-bottom:1px solid #0f172a">
              <td style="padding:10px 14px;color:#64748b">{medal[i]}</td>
              <td style="padding:10px 14px;color:#e2e8f0;font-weight:600">{row[name_col]}</td>
              <td style="padding:10px 14px;color:#60a5fa;font-weight:700">{row['CGPA']}</td>
              <td style="padding:10px 14px;color:{gc};font-weight:700">{row['Grade']}</td>
              <td style="padding:10px 14px;color:{rc2};font-weight:600">{row['Result']}</td>
              <td style="padding:10px 14px;color:#94a3b8">{int(row['Overall_Total_650'])}/650</td>
            </tr>"""

        display(HTML(f"""
        <div style="margin-top:20px">
          <div class="section-title" style="margin-bottom:10px">🏆 TOP 10 LEADERBOARD</div>
          <table style="width:100%;border-collapse:collapse;
                        background:linear-gradient(135deg,#1e293b,#0f172a);
                        border-radius:12px;overflow:hidden;
                        font-family:'Rajdhani',sans-serif;font-size:1em">
            <thead><tr style="background:#0f172a;color:#64748b;letter-spacing:1px;font-size:0.82em">
              <th style="padding:12px 14px;text-align:left">#</th>
              <th style="padding:12px 14px;text-align:left">NAME</th>
              <th style="padding:12px 14px;text-align:left">CGPA</th>
              <th style="padding:12px 14px;text-align:left">GRADE</th>
              <th style="padding:12px 14px;text-align:left">RESULT</th>
              <th style="padding:12px 14px;text-align:left">TOTAL</th>
            </tr></thead>
            <tbody>{rows}</tbody>
          </table>
        </div>"""))

        # ── Student Search ──
        display(HTML('<div class="section-title" style="margin-top:20px">🔍 STUDENT SEARCH</div>'))
        search_box = widgets.Text(placeholder='Type student name...',
                                  description='Search:',
                                  layout=widgets.Layout(width='380px'))
        search_out = widgets.Output()

        def do_search(change):
            with search_out:
                clear_output()
                query = change['new'].strip().lower()
                if not query: return
                matches = df[df[name_col].astype(str).str.lower().str.contains(query)]
                if matches.empty:
                    display(HTML('<p style="color:#f87171;font-family:monospace">No student found.</p>'))
                    return
                for _, row in matches.iterrows():
                    gc  = {"O":"#34d399","A+":"#60a5fa","A":"#a78bfa",
                           "B+":"#fb923c","B":"#fbbf24","F":"#f87171"}.get(str(row.get("Grade","F")),"white")
                    rc2 = "#34d399" if row.get("Result","Fail")=="Pass" else "#f87171"
                    subj_rows = ""
                    sem2_c = clean_marks(sem2).fillna(0)
                    for col in sem2_subjects:
                        fin_col = col+"_Final_70"; tot_col = col+"_Total_100"
                        if fin_col in row and tot_col in row:
                            int_val = sem2_c[col].iloc[df[df[name_col]==row[name_col]].index[0]] if len(df[df[name_col]==row[name_col]])>0 else 'N/A'
                            subj_rows += f"""<tr style="border-bottom:1px solid #0f172a">
                              <td style="padding:6px 12px;color:#94a3b8;font-size:0.85em">{col[:35]}</td>
                              <td style="padding:6px 12px;color:#60a5fa">{int_val}</td>
                              <td style="padding:6px 12px;color:#34d399">{row[fin_col]:.1f}/70</td>
                              <td style="padding:6px 12px;color:#fbbf24;font-weight:700">{row[tot_col]:.1f}/100</td>
                            </tr>"""
                    for col in lab_subjects:
                        fin_col = col+"_Final_35"; tot_col = col+"_Total_50"
                        if fin_col in row and tot_col in row:
                            subj_rows += f"""<tr style="border-bottom:1px solid #0f172a">
                              <td style="padding:6px 12px;color:#94a3b8;font-size:0.85em">{col[:35]}</td>
                              <td style="padding:6px 12px;color:#60a5fa">N/A</td>
                              <td style="padding:6px 12px;color:#34d399">{row[fin_col]:.1f}/35</td>
                              <td style="padding:6px 12px;color:#fbbf24;font-weight:700">{row[tot_col]:.1f}/50</td>
                            </tr>"""
                    display(HTML(f"""
                    <div style="background:linear-gradient(135deg,#1e293b,#0f172a);
                                border-radius:14px;padding:20px;
                                border:1px solid rgba(251,146,60,0.25);margin:10px 0">
                      <h3 style="font-family:'Orbitron',monospace;color:#fb923c;margin:0 0 14px 0;font-size:1em">
                        👤 {row[name_col]}
                      </h3>
                      <div style="display:flex;gap:18px;flex-wrap:wrap;font-family:'Rajdhani',sans-serif;margin-bottom:14px">
                        <div><span style="color:#64748b">CGPA: </span><span style="color:#60a5fa;font-size:1.5em;font-weight:700">{row.get('CGPA','N/A')}</span></div>
                        <div><span style="color:#64748b">Grade: </span><span style="color:{gc};font-size:1.5em;font-weight:700">{row.get('Grade','N/A')}</span></div>
                        <div><span style="color:#64748b">Result: </span><span style="color:{rc2};font-size:1.3em;font-weight:700">{row.get('Result','N/A')}</span></div>
                        <div><span style="color:#64748b">Total: </span><span style="color:#fbbf24;font-size:1.3em;font-weight:700">{int(row.get('Overall_Total_650',0))}/650</span></div>
                      </div>
                      <table style="width:100%;border-collapse:collapse;font-family:'Rajdhani',sans-serif;font-size:0.9em">
                        <thead><tr style="background:#0f172a;color:#64748b;font-size:0.8em;letter-spacing:1px">
                          <th style="padding:8px 12px;text-align:left">SUBJECT</th>
                          <th style="padding:8px 12px;text-align:left">INTERNAL</th>
                          <th style="padding:8px 12px;text-align:left">PREDICTED FINAL</th>
                          <th style="padding:8px 12px;text-align:left">TOTAL</th>
                        </tr></thead>
                        <tbody>{subj_rows}</tbody>
                      </table>
                    </div>"""))

        search_box.observe(do_search, names='value')
        display(search_box, search_out)

        # ── What-If Simulator ──
        display(HTML('<div class="section-title" style="margin-top:20px">🎯 WHAT-IF SCORE SIMULATOR</div>'))
        sim_slider = widgets.IntSlider(value=20, min=0, max=30, step=1,
                                       description='Internal:',
                                       style={'description_width':'80px'},
                                       layout=widgets.Layout(width='420px'))
        sim_out = widgets.Output()

        def simulate(change):
            with sim_out:
                clear_output()
                val = change['new']
                try:
                    X3s  = pd.DataFrame({"Internal":[val], "Delta":[0]})
                    pred = float(np.clip(M3_model.predict(X3s)[0], 0, 70))
                    total= val + pred
                    cgpa_e  = round((total/100)*10, 2)
                    g_e     = grade(cgpa_e)
                    gc2     = {"O":"#34d399","A+":"#60a5fa","A":"#a78bfa",
                               "B+":"#fb923c","B":"#fbbf24","F":"#f87171"}.get(g_e,"white")
                    result_e= "Pass" if cgpa_e >= 5 else "Fail"
                    rc3     = "#34d399" if result_e=="Pass" else "#f87171"
                    display(HTML(f"""
                    <div style="background:linear-gradient(135deg,#1e293b,#0f172a);
                                border-radius:10px;padding:16px 20px;
                                border:1px solid rgba(251,146,60,0.2);
                                font-family:'Rajdhani',sans-serif;font-size:1.05em;
                                display:flex;gap:20px;flex-wrap:wrap;align-items:center">
                      <div><span style="color:#64748b">Internal: </span>
                           <span style="color:#60a5fa;font-weight:700;font-size:1.2em">{val}/30</span></div>
                      <div>→</div>
                      <div><span style="color:#64748b">Predicted Final: </span>
                           <span style="color:#34d399;font-weight:700;font-size:1.2em">{pred:.1f}/70</span></div>
                      <div><span style="color:#64748b">Total: </span>
                           <span style="color:#fbbf24;font-weight:700;font-size:1.2em">{total:.1f}/100</span></div>
                      <div><span style="color:#64748b">Grade: </span>
                           <span style="color:{gc2};font-weight:700;font-size:1.2em">{g_e}</span></div>
                      <div><span style="color:#64748b">Result: </span>
                           <span style="color:{rc3};font-weight:700;font-size:1.2em">{result_e}</span></div>
                    </div>"""))
                except Exception as e:
                    display(HTML(f'<p style="color:#f87171;font-family:monospace">Train model first! {e}</p>'))

        sim_slider.observe(simulate, names='value')
        display(sim_slider, sim_out)

dash_btn.on_click(show_dashboard)
display(dash_btn, dash_out)

display(HTML('<div class="section-title">📊 STEP 5 — Model Performance Metrics</div>'))

metrics_btn = widgets.Button(description='📊 Show Metrics',
    style={'button_color': '#6d28d9'},
    layout=widgets.Layout(width='200px', height='44px'))
metrics_out = widgets.Output()

def show_metrics(b):
    with metrics_out:
        clear_output()
        if M3_model is None:
            display(HTML('<p style="color:#f87171;font-family:monospace">❌ Train model first!</p>'))
            return

        # ── Metrics Cards ──
        display(HTML(f"""
        <div class="cards">
          <div class="card"><div class="num" style="color:#fb923c">{model_mae:.2f}</div><div class="lbl">MAE</div></div>
          <div class="card"><div class="num" style="color:#fbbf24">{model_rmse:.2f}</div><div class="lbl">RMSE</div></div>
          <div class="card"><div class="num" style="color:#60a5fa">{model_r2:.4f}</div><div class="lbl">R² SCORE</div></div>
          <div class="card"><div class="num" style="color:#34d399">{model_acc:.1f}%</div><div class="lbl">ACCURACY</div></div>
        </div>"""))

        # ── Metrics Table ──
        data = {
            "Metric":      ["MAE",  "RMSE",  "R² Score",  "Accuracy (%)"],
            "Value":       [round(model_mae,2), round(model_rmse,2),
                            round(model_r2,4),  round(model_acc,2)],
            "Formula":     [
                "MAE = (1/n) Σ |Actual − Predicted|",
                "RMSE = √[(1/n) Σ (Actual − Predicted)²]",
                "R² = 1 − (SS_res / SS_tot)",
                "Accuracy = (1 − MAE/MaxMarks) × 100"
            ],
            "Interpretation": [
                "Average prediction error in marks",
                "Penalises large errors more",
                "How well model fits data (1.0 = perfect)",
                "Overall prediction accuracy"
            ]
        }
        df_metrics = pd.DataFrame(data)

        rows = ""
        for _, row in df_metrics.iterrows():
            rows += f"""<tr style="border-bottom:1px solid #0f172a">
              <td style="padding:10px 14px;color:#fb923c;font-weight:700">{row['Metric']}</td>
              <td style="padding:10px 14px;color:#34d399;font-weight:700">{row['Value']}</td>
              <td style="padding:10px 14px;color:#94a3b8;font-size:0.85em">{row['Formula']}</td>
              <td style="padding:10px 14px;color:#cbd5e1;font-size:0.85em">{row['Interpretation']}</td>
            </tr>"""

        display(HTML(f"""
        <table style="width:100%;border-collapse:collapse;
                      background:linear-gradient(135deg,#1e293b,#0f172a);
                      border-radius:12px;overflow:hidden;
                      font-family:'Rajdhani',sans-serif;font-size:0.95em;margin-top:14px">
          <thead><tr style="background:#0f172a;color:#64748b;letter-spacing:1px;font-size:0.82em">
            <th style="padding:12px 14px;text-align:left">METRIC</th>
            <th style="padding:12px 14px;text-align:left">VALUE</th>
            <th style="padding:12px 14px;text-align:left">FORMULA</th>
            <th style="padding:12px 14px;text-align:left">INTERPRETATION</th>
          </tr></thead>
          <tbody>{rows}</tbody>
        </table>"""))

        # ── Model Info Table ──
        display(HTML("""
        <div style="margin-top:20px">
          <div class="section-title" style="margin-bottom:10px">🧠 MODEL DETAILS</div>
          <table style="width:100%;border-collapse:collapse;
                        background:linear-gradient(135deg,#1e293b,#0f172a);
                        border-radius:12px;overflow:hidden;
                        font-family:'Rajdhani',sans-serif;font-size:0.95em">
            <tbody>
              <tr style="border-bottom:1px solid #0f172a">
                <td style="padding:10px 14px;color:#64748b;width:220px">Model Name</td>
                <td style="padding:10px 14px;color:#fb923c;font-weight:700">Temporal Student Performance Progression Model</td>
              </tr>
              <tr style="border-bottom:1px solid #0f172a">
                <td style="padding:10px 14px;color:#64748b">Algorithm</td>
                <td style="padding:10px 14px;color:#e2e8f0">Gradient Boosted Decision Trees (GBDT)</td>
              </tr>
              <tr style="border-bottom:1px solid #0f172a">
                <td style="padding:10px 14px;color:#64748b">Key Feature</td>
                <td style="padding:10px 14px;color:#e2e8f0">Performance Delta (Final − Internal) = progression tracking</td>
              </tr>
              <tr style="border-bottom:1px solid #0f172a">
                <td style="padding:10px 14px;color:#64748b">Why Final Year Level</td>
                <td style="padding:10px 14px;color:#e2e8f0">Learns student improvement trend — not static prediction</td>
              </tr>
              <tr style="border-bottom:1px solid #0f172a">
                <td style="padding:10px 14px;color:#64748b">n_estimators</td>
                <td style="padding:10px 14px;color:#e2e8f0">150 trees</td>
              </tr>
              <tr style="border-bottom:1px solid #0f172a">
                <td style="padding:10px 14px;color:#64748b">Learning Rate</td>
                <td style="padding:10px 14px;color:#e2e8f0">0.05</td>
              </tr>
              <tr>
                <td style="padding:10px 14px;color:#64748b">Learning Type</td>
                <td style="padding:10px 14px;color:#e2e8f0">Supervised — Regression</td>
              </tr>
            </tbody>
          </table>
        </div>"""))

metrics_btn.on_click(show_metrics)
display(metrics_btn, metrics_out)

display(HTML('<div class="section-title">⬇️ STEP 6 — Download Results</div>'))

dl_btn = widgets.Button(description='⬇️ Download CSV',
    style={'button_color': '#b45309'},
    layout=widgets.Layout(width='200px', height='44px'))
dl_out = widgets.Output()

def download_all(b):
    with dl_out:
        clear_output()
        for fname in ["M3_Temporal_Predictions.csv", "M3_dashboard.png"]:
            try:
                files.download(fname)
                display(HTML(f'<p style="color:#34d399;font-family:monospace">⬇️ {fname} downloading...</p>'))
            except Exception as e:
                display(HTML(f'<p style="color:#f87171;font-family:monospace">❌ {fname}: {e}</p>'))

dl_btn.on_click(download_all)
display(dl_btn, dl_out)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 40.6 MB/s eta 0:00:00
✅ All libraries loaded!


✅ Subjects and helpers defined!


Button(description='🚀 Train Model 3', layout=Layout(height='44px', width='220px'), style=ButtonStyle(button_co…

Output()

Button(description='⚡ Run Predictions', layout=Layout(height='44px', width='220px'), style=ButtonStyle(button_…

...

Output()

📂 Choose sem1.csv ↓


Saving sem1.csv to sem1.csv
📂 Choose sem1.csv ↓


Saving sem 2.csv to sem 2.csv
📂 Choose sem1.csv ↓


📂 Choose sem1.csv ↓
